# Modular multiplication

Adapted from Qualtran v0.7.0, Copyright 2024 Google LLC, Apache-2.0.
The examples below pair declared resource graphs with the emitted FlagQuantum circuits.

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.modular_multiplication_catalog import modular_multiplication_examples
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

examples = modular_multiplication_examples()

def display_signatures(*names):
    display(SVG(field_signature_svg([(name, examples[name].signature(), {}) for name in names])))

def display_calls(name):
    graph = examples[name].call_graph()
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    display(graph.leaf_counts)

def emitted_counts(circuit):
    counts = Counter(gate.name for gate in circuit.to_ir().instructions)
    return {"T/T†": counts["t"] + counts["tdg"], "measure": counts["measure"],
            "CX": counts["cx"], "qubits": circuit.n_qubits}

## `ModDbl`
An n-bit modular doubling gate.

Implements $\ket{x} \rightarrow \ket{2x \mod p}$ using $2n$ Toffoli gates.

#### Parameters
 - `dtype`: Dtype of the number to double.
 - `p`: The modulus for the doubling. 

#### Registers
 - `x`: The register containing the number to double. 

#### References
 - [How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates](https://arxiv.org/abs/2306.08585). Fig 6d and 8


In [ ]:
from examples.algorithms.register_modular_doubling import ModularDoubling
from flagquantum.compiler.registers.modular.modular_doubling import modular_double_requirements

### Example Instances

In [ ]:
small_double = examples["moddbl_small"]

In [ ]:
large_double = examples["moddbl_large"]

#### Graphical Signature

In [ ]:
display_signatures("moddbl_small", "moddbl_large")

### Call Graph

In [ ]:
display_calls("moddbl_small")

## `CModMulK`
Perform controlled modular multiplication by a constant.

Applies $\ket{c}\ket{x} \rightarrow \ket{c} \ket{x*k^c \mod p}$.

#### Parameters
 - `dtype`: Dtype of the register.
 - `k`: The integer multiplicative constant.
 - `mod`: The integer modulus. 

#### Registers
 - `ctrl`: The control bit
 - `x`: The integer being multiplied


In [ ]:
from examples.algorithms.register_modular_multiplication import ModularMultiplication

### Example Instances

In [ ]:
symbolic_multiply = examples["modmul_symb"]

In [ ]:
constant_multiply = examples["modmul"]

#### Graphical Signature

In [ ]:
display_signatures("modmul_symb", "modmul")

### Call Graph

In [ ]:
display_calls("modmul_symb")

## `DirtyOutOfPlaceMontgomeryModMul`
Perform windowed montgomery modular multiplication.

Applies the trasformation
$$
    \ket{x}\ket{y}\ket{0}\ket{0}\ket{0} \rightarrow \ket{x}\ket{y}\ket{xy2^{-n}}\ket{h}\ket{c}
$$

Where:

- $n$ is the bitsize.
- $x, y$ are in montgomery form
- $h$ is an ancilla register that represents intermidate values.
- $c$ is whether a final modular reduction was applied or not.

#### Parameters
 - `bitsize`: size of the numbers.
 - `window_size`: size of the window.
 - `mod`: The integer modulus.
 - `uncompute`: whether to compute or uncompute. 

#### Registers
 - `x`: The first integer
 - `y`: The second integer
 - `target`: product in montgomery form $xy 2^{-n}$
 - `qrom_indices`: concatination of the indicies used to query QROM.
 - `reduced`: whether a final modular reduction was applied. 

#### References
 - [Performance Analysis of a Repetition Cat Code Architecture: Computing 256-bit Elliptic Curve Logarithm in 9 Hours with 126 133 Cat Qubits](https://arxiv.org/abs/2302.06639).     Appendix C4.
 - [How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates](https://arxiv.org/abs/2306.08585).     page 8.


In [ ]:
from examples.algorithms.register_windowed_mod_mul import WindowedMontgomeryMultiplication

### Example Instances

In [ ]:
small_windowed = examples["dirtyoutofplacemontgomerymodmul_small"]

In [ ]:
medium_windowed = examples["dirtyoutofplacemontgomerymodmul_medium"]

#### Graphical Signature

In [ ]:
display_signatures("dirtyoutofplacemontgomerymodmul_small", "dirtyoutofplacemontgomerymodmul_medium")

### Call Graph

In [ ]:
display_calls("dirtyoutofplacemontgomerymodmul_small")

### Emit the native circuits

The concrete doubling and controlled multiplication circuits accept valid residues. Doubling returns a new logical wire order. The windowed circuit retains the lookup indices and reduction flag. Its adjoint consumes the actual source circuit image. The locked source's final gate-stage addition and high-level subtraction can disagree when reduction is needed. The source call graphs above describe direct children. The counts below come from compiled primitive instructions. The windowed implementation clears the table by reversing the forward lookup, which has a different cost from the abstract measurement-based adjoint model.

In [ ]:
for example in (small_double, large_double):
    n = example.width
    work, _ = modular_double_requirements(n)
    circuit = DynamicCircuit(n + work)
    op = ModularDoubling(tuple(range(n)), tuple(range(n, n + work)), modulus=example.modulus)
    circuit.append(op)
    print(example.kind, n, emitted_counts(circuit), "result wires:", op.outputs[0])

example = constant_multiply
op = ModularMultiplication(example.constant, width=example.width, modulus=example.modulus)
circuit = DynamicCircuit(op.qubit_count).append(op)
print(example.kind, emitted_counts(circuit))
print("Source model T (4T CSwap):", 40 * example.width**2 + 20 * example.width)

for example in (small_windowed, medium_windowed):
    params = {"width": example.width, "window": example.window_size, "modulus": example.modulus}
    windowed = WindowedMontgomeryMultiplication(**params)
    circuit = DynamicCircuit(windowed.qubit_count).append(windowed)
    counts = emitted_counts(circuit)
    print(example.kind, counts, "result:", windowed.target_wires,
          "indices:", windowed.history_wires, "reduction:", windowed.reduction_wire)
    undo = WindowedMontgomeryMultiplication(adjoint=True, **params)
    circuit.append(undo)
    assert emitted_counts(circuit)["T/T†"] == 2 * counts["T/T†"]